In [1]:
import pandas as pd
import geopandas as gpd



In [2]:
stations_gdf = gpd.read_file("../raw/velib-emplacement-des-stations.geojson")
quartiers_gdf = gpd.read_file("../raw/quartier_paris.geojson")

In [3]:
stations_gdf.head()

,stationcode,name,capacity,station_opening_hours,geometry
0,11104,Charonne - Robert et Sonia Delaunay,20,None,POINT (2.39257 48.85591)
1,22207,Rond-point du Dr Albert Schweitzer,22,None,POINT (2.31439 48.78981)
2,44010,8 Mai 1945 - 10 Juillet 1940,30,None,POINT (2.3979 48.78457)
3,31009,Place Du Général De Gaulle,26,None,POINT (2.43299 48.86876)
4,21312,Lucie Aubrac - Fort,30,None,POINT (2.27118 48.81814)


In [4]:
quartiers_gdf.head()

,n_sq_qu,c_qu,c_quinsee,l_qu,c_ar,n_sq_ar,perimetre,surface,geom_x_y,st_area_shape,st_perimeter_shape,geometry
0,750000022,22,7510602,Odéon,6,750000006,3516.314464,7.161484e+05,"{'lon': 2.3363388275876775, 'lat': 48.84780062...",7.161484e+05,3516.110384,"POLYGON ((2.33699 48.8529, 2.33894 48.85242, 2..."
1,750000043,43,7511103,Roquette,11,750000011,4973.010557,1.172087e+06,"{'lon': 2.380364061726767, 'lat': 48.857064040...",1.172087e+06,4972.768946,"POLYGON ((2.37972 48.85344, 2.37937 48.85339, ..."
2,750000023,23,7510603,Notre-Dame-des-Champs,6,750000006,4559.989773,8.613070e+05,"{'lon': 2.327356878234607, 'lat': 48.846427594...",8.613070e+05,4559.726714,"POLYGON ((2.33676 48.84013, 2.33673 48.83965, ..."
3,750000030,30,7510802,Faubourg-du-Roule,8,750000008,3773.673073,7.965891e+05,"{'lon': 2.3041188097211407, 'lat': 48.87413557...",7.965891e+05,3773.554585,"POLYGON ((2.31197 48.86993, 2.31011 48.86898, ..."
4,750000057,57,7511501,Saint-Lambert,15,750000015,6928.792072,2.829202e+06,"{'lon': 2.296919974448867, 'lat': 48.834293628...",2.829202e+06,6928.319925,"POLYGON ((2.30425 48.84041, 2.30497 48.84061, ..."


In [5]:
quartiers_gdf = quartiers_gdf.to_crs(stations_gdf.crs)

In [6]:
stations_quartiers = gpd.sjoin(
    stations_gdf,
    quartiers_gdf,
    how="inner",
    predicate="within"
)

In [7]:
stations_quartiers.head()

,stationcode,name,capacity,station_opening_hours,geometry,index_right,n_sq_qu,c_qu,c_quinsee,l_qu,c_ar,n_sq_ar,perimetre,surface,geom_x_y,st_area_shape,st_perimeter_shape
0,11104,Charonne - Robert et Sonia Delaunay,20,None,POINT (2.39257 48.85591),14,750000044,44,7511104,Sainte-Marguerite,11,750000011,4591.310799,9.296092e+05,"{'lon': 2.3887648336025635, 'lat': 48.85209650...",9.296092e+05,4591.073927
5,15133,Saint Lambert - Blomet,27,None,POINT (2.29306 48.83659),4,750000057,57,7511501,Saint-Lambert,15,750000015,6928.792072,2.829202e+06,"{'lon': 2.296919974448867, 'lat': 48.834293628...",2.829202e+06,6928.319925
7,8050,Boétie - Ponthieu,45,None,POINT (2.30768 48.87142),3,750000030,30,7510802,Faubourg-du-Roule,8,750000008,3773.673073,7.965891e+05,"{'lon': 2.3041188097211407, 'lat': 48.87413557...",7.965891e+05,3773.554585
8,15010,Square Cambronne,59,None,POINT (2.30257 48.84757),9,750000058,58,7511502,Necker,15,750000015,5979.711469,1.578484e+06,"{'lon': 2.3107774536393966, 'lat': 48.84271125...",1.578484e+06,5979.347771
11,17122,Bridaine - Lamandé,29,None,POINT (2.32062 48.88621),41,750000067,67,7511703,Batignolles,17,750000017,5832.690026,1.441711e+06,"{'lon': 2.313856169006357, 'lat': 48.888481513...",1.441711e+06,5832.584353


In [8]:
stations_quartiers = stations_quartiers[[
    "stationcode",
    "name",
    "capacity",
    "geometry",
    "c_qu",
    "l_qu"
]]

In [9]:
stations_quartiers.to_parquet("../silver/silver_velib_quartiers.parquet")

In [10]:
df_aggregated = (stations_quartiers.groupby("l_qu").agg(velib_station=("stationcode","count"),capacity=("capacity","sum")).reset_index())

In [12]:
df_silver = pd.read_parquet("../silver/silver_velib_quartiers.parquet")

In [16]:
df_aggregated = (df_silver.groupby(["l_qu","c_qu"]).agg(velib_station=("stationcode","count"),capacity=("capacity","sum")).reset_index())

In [17]:
df_aggregated

,l_qu,c_qu,velib_station,capacity
0,Amérique,75,17,478
1,Archives,11,4,129
2,Arsenal,15,7,175
3,Arts-et-Métiers,9,4,123
4,Auteuil,61,19,769
...,...,...,...,...
75,Sorbonne,20,11,394
76,Ternes,65,17,598
77,Val-de-Grace,19,6,182
78,Villette,73,13,339


In [18]:
df_aggregated.to_parquet("../gold/capacity_velib_quartier.parquet")